# `future` vs `pymfiga`: matrix-free elasticity on the Kirsch benchmark

Same problem as `12_plate_with_hole.ipynb` / `16_plate_with_hole_matrix_free.ipynb`:
a quarter annulus (`Rin=1, Rex=2`) under the exact Kirsch (1898) traction on its outer
arc, symmetry boundary conditions on the two straight edges, plane stress
(`E=1e3, nu=0.3`). This notebook compares two *matrix-free* weighted-quadrature (WQ)
implementations of the stiffness operator against each other and against a classical
Gauss-quadrature + direct-solve reference, all evaluated against the true closed-form
Kirsch displacement field -- gradient included, **analytically**, not by finite
difference.

Three pipelines:

- **`future`-Gauss** -- `PatchIntegrator.integrate_stiffness()` (classical Gauss
  quadrature) + `scipy.sparse.linalg.spsolve`. The accuracy reference.
- **`future`-MF** -- `WQMatrixFreeStiffness` (this repo's C++ matrix-free WQ operator)
  + a fast-diagonalization preconditioner + GMRES.
- **`pymfiga`-MF** -- the vendored `pymfiga` package's own matrix-free WQ operator
  (`MechanicalModel.compute_mf_stiffness`) + its own fast-diagonalization
  preconditioner + the *same* GMRES driver as `future`-MF (so the only thing that
  differs between the two matrix-free rows is the operator/preconditioner
  implementation itself, not the outer iterative solver).

In [ ]:
import os

# Pin every native thread pool to 1 BEFORE numpy/scipy/yeti_iga are imported --
# these libraries read the thread count once, at first use -- so assembly/solve
# timings below are comparable across pipelines (same convention as
# kirsch_hpc_sweep/kirsch_sweep.py).
for _env in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_env] = "1"

import logging
# pymfiga logs an INFO block (quadrature/geometry/material/boundary/preconditioner
# details) on every single model construction -- harmless, but with 3 pipelines x
# 21 (degree, nbel) cases in the sweep below it drowns the notebook's own output.
logging.disable(logging.CRITICAL)

import time

import numpy as np
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (
    BSpline, BSplineSurface, ControlPointManager, Patch as FPatch,
    GlobalDOFManager, PatchDOFManager, Material, PlaneStress,
    IGABasis1D, PatchIntegrator, Traction, ScalarLocalOperator,
    WQMatrixFreeStiffness,
)
from yeti_iga.future.fast_diagonalization import FastDiagonalizationPreconditioner
from yeti_iga.future.matrix_free_solver import solve as mf_solve, free_dof_operator

from yeti_iga.pymfiga.common.material import LinearElasticity
from yeti_iga.pymfiga.iga.geometry import GeomdlGenerator, SinglePatch
from yeti_iga.pymfiga.iga.boundary import BoundaryCondition, ParametricDirection, BoundarySide, DOF
from yeti_iga.pymfiga.iga.single_model import MechanicalModel

# Kirsch benchmark parameters (examples/future/12_plate_with_hole.ipynb)
Rin, Rex, Tx = 1.0, 2.0, 1.0
E, nu = 1e3, 0.3

# pymfiga's NURBS arc builder asserts degree > 1 (a degree-1 rational curve is a
# polyline, not a circle): every pipeline here needs degree >= 2.
BASE_DEGREE = 2

## Problem setup

Quarter annulus, parametric direction 0 (`XI`) = radial, direction 1 (`ETA`) =
angular (confirmed from `pymfiga`'s own `nurbs_quarter_annulus` geometry builder).
Boundary conditions are symmetry conditions, not a full clamp:

- `ETA`-min (`theta=0`, the x-axis) blocks `u_y` only;
- `ETA`-max (`theta=90 deg`, the y-axis) blocks `u_x` only;
- the exact Kirsch traction is applied on the outer arc, `XI`-max (`r=Rex`).

The exact solution (`kirsch_displacement`) and its gradient
(`kirsch_displacement_gradient`) are used both to build the traction and to measure
the L2/H1 error of every pipeline's discrete solution.

In [ ]:
class KirschTraction(Traction):
    """Exact Kirsch traction at the outer arc of the quarter annulus."""

    def __init__(self, Rin, Tx=1.0):
        super().__init__()
        self.Rin = Rin
        self.Tx = Tx

    def evaluate(self, pt):
        x, y = pt[0], pt[1]
        r = np.sqrt(x**2 + y**2)
        theta = np.arctan2(y, x)
        R2 = (self.Rin / r) ** 2
        R4 = (self.Rin / r) ** 4
        g1 = (self.Tx / 2) * (
            2 * np.cos(theta)
            - R2 * (2 * np.cos(theta) + 3 * np.cos(3 * theta))
            + 3 * R4 * np.cos(3 * theta)
        )
        g2 = (3 * self.Tx / 2) * np.sin(3 * theta) * (R4 - R2)
        return np.array([g1, g2])


def kirsch_surface_force(Rin, Tx):
    """pymfiga's `assemble_surface_force` convention: a callable receiving
    args['position'] of shape (2, n_pts) and returning a (2, n_pts) force
    array in the SAME global-Cartesian convention as future's Traction."""

    def fn(args):
        position = np.asarray(args["position"])
        x, y = position[0], position[1]
        r = np.sqrt(x**2 + y**2)
        theta = np.arctan2(y, x)
        R2 = (Rin / r) ** 2
        R4 = (Rin / r) ** 4
        g1 = (Tx / 2) * (
            2 * np.cos(theta)
            - R2 * (2 * np.cos(theta) + 3 * np.cos(3 * theta))
            + 3 * R4 * np.cos(3 * theta)
        )
        g2 = (3 * Tx / 2) * np.sin(3 * theta) * (R4 - R2)
        return np.vstack([g1, g2])

    return fn


def kirsch_displacement(pt, Rin, Tx, E, nu):
    x, y = pt[0], pt[1]
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    a2 = Rin**2
    u_x = Tx / (2 * E) * (2 * (r + 2 * a2 / r) * np.cos(theta)
                          + (1 + nu) * a2 / r * (1 - a2 / r**2) * np.cos(3 * theta))
    u_y = Tx / (2 * E) * (-2 * (nu * r + (1 - nu) * a2 / r) * np.sin(theta)
                          + (1 + nu) * a2 / r * (1 - a2 / r**2) * np.sin(3 * theta))
    return np.array([u_x, u_y])


def kirsch_displacement_gradient(pt, Rin, Tx, E, nu):
    """Analytic gradient of kirsch_displacement, via the chain rule on (r, theta).

    Deliberately analytical, not finite-difference: a finite-difference gradient's
    own noise floor (~eps**(2/3), regardless of step size) would mask the true
    convergence of the H1 error once the degree sweep below goes high enough for
    the discretization error to drop below that floor.

    u_x = C*(A(r) cos(theta) + B(r) cos(3 theta))
    u_y = C*(D(r) sin(theta) + B(r) sin(3 theta))
    with C = Tx/(2E), a2 = Rin**2,
    A(r) = 2r + 4 a2/r, D(r) = -2 nu r - 2(1-nu) a2/r,
    B(r) = (1+nu)(a2/r - a2**2/r**3).
    """
    x, y = pt[0], pt[1]
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)
    a2 = Rin**2
    C = Tx / (2 * E)
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    cos_3t, sin_3t = np.cos(3 * theta), np.sin(3 * theta)

    A = 2 * r + 4 * a2 / r
    Ap = 2 - 4 * a2 / r**2
    B = (1 + nu) * (a2 / r - a2**2 / r**3)
    Bp = (1 + nu) * (3 * a2**2 / r**4 - a2 / r**2)
    D = -2 * nu * r - 2 * (1 - nu) * a2 / r
    Dp = -2 * nu + 2 * (1 - nu) * a2 / r**2

    dux_dr = C * (Ap * cos_t + Bp * cos_3t)
    dux_dth = C * (-A * sin_t - 3 * B * sin_3t)
    duy_dr = C * (Dp * sin_t + Bp * sin_3t)
    duy_dth = C * (D * cos_t + 3 * B * cos_3t)

    # chain rule: df/dx = f_r cos(theta) - f_theta sin(theta)/r
    #             df/dy = f_r sin(theta) + f_theta cos(theta)/r
    dux_dx = dux_dr * cos_t - dux_dth * sin_t / r
    dux_dy = dux_dr * sin_t + dux_dth * cos_t / r
    duy_dx = duy_dr * cos_t - duy_dth * sin_t / r
    duy_dy = duy_dr * sin_t + duy_dth * cos_t / r

    return np.array([[dux_dx, dux_dy], [duy_dx, duy_dy]])

In [ ]:
class L2ErrorOperator(ScalarLocalOperator):
    """Integrand |u_h - u_ex|^2 for the L2 error norm."""

    def __init__(self, u_exact_fn):
        super().__init__()
        self.u_exact_fn = u_exact_fn

    def compute_scalar_integrand(self, R, dRdx, dRdy, physical_point, u_local):
        R = np.asarray(R)
        u_x, u_y = np.asarray(u_local)[0::2], np.asarray(u_local)[1::2]
        u_ex = self.u_exact_fn(physical_point)
        return (R @ u_x - u_ex[0]) ** 2 + (R @ u_y - u_ex[1]) ** 2


class H1SemiNormErrorOperator(ScalarLocalOperator):
    """Integrand for the H1 semi-norm error (gradient mismatch)."""

    def __init__(self, grad_exact_fn):
        super().__init__()
        self.grad_exact_fn = grad_exact_fn

    def compute_scalar_integrand(self, R, dRdx, dRdy, physical_point, u_local):
        dRdx, dRdy = np.asarray(dRdx), np.asarray(dRdy)
        u_x, u_y = np.asarray(u_local)[0::2], np.asarray(u_local)[1::2]
        grad_ex = self.grad_exact_fn(physical_point)
        return (
            (dRdx @ u_x - grad_ex[0, 0]) ** 2 + (dRdy @ u_x - grad_ex[0, 1]) ** 2
            + (dRdx @ u_y - grad_ex[1, 0]) ** 2 + (dRdy @ u_y - grad_ex[1, 1]) ** 2
        )

## Geometry: one patch, shared bit-for-bit between both codebases

`pymfiga`'s `GeomdlGenerator`/`SinglePatch` builds the quarter annulus once; `future`'s
patch is then built by copying its control points, weights and knot vectors
point-for-point (`build_future_from_pymfiga`) -- not an independent reconstruction.
This is what makes the two matrix-free pipelines below a fair comparison: any
difference in their results comes from the operator/preconditioner implementation,
never from a geometry mismatch.

In [ ]:
def build_pymfiga_patch(degree, nbel, Rin, Rex):
    geo_args = {"degree": degree, "nbel": nbel, "parameters": {"Rin": Rin, "Rex": Rex}}
    geometry = GeomdlGenerator(filename="nurbs_quarter_annulus", geo_args=geo_args).export_geometry()
    pp = SinglePatch(geometry, quadclass="wq")
    pp.generate()
    return pp


def build_future_from_pymfiga(pp):
    deg_u, deg_v = int(pp.degree[0]), int(pp.degree[1])
    kv_u, kv_v = pp.knotvector[0], pp.knotvector[1]
    nu_, nv_ = int(pp.nbctrlpts[0]), int(pp.nbctrlpts[1])
    ctrlpts, weights = pp.ctrlpts, pp.nurbs_weights  # (2, n), XI-fastest

    mgr = ControlPointManager(dim=2)
    for i in range(nu_ * nv_):
        mgr.add_point([float(ctrlpts[0, i]), float(ctrlpts[1, i])], float(weights[i]))

    su, sv = BSpline(deg_u, np.asarray(kv_u)), BSpline(deg_v, np.asarray(kv_v))
    patch0 = FPatch(BSplineSurface(su, sv), mgr, list(range(nu_ * nv_)), [nu_, nv_])
    n_cp = nu_ * nv_
    gdm = GlobalDOFManager([2] * n_cp)
    pdm = PatchDOFManager(2, list(range(n_cp)), gdm)
    pk = FPatch(patch0.tensor, patch0.cp_manager, list(patch0.global_indices), list(patch0.local_shape), pdm)
    return pk, pdm


def future_fixed_dofs(pk, pdm):
    """Symmetry BCs: ETA-min (theta=0, x-axis) blocks u_y, ETA-max (theta=90deg,
    y-axis) blocks u_x -- not a one-sided full clamp."""
    fixed = set()
    for cp in pk.boundary_control_points(1, 0):
        fixed.add(pdm.get_global_dof_indices(cp)[1])
    for cp in pk.boundary_control_points(1, 1):
        fixed.add(pdm.get_global_dof_indices(cp)[0])
    return np.array(sorted(fixed))

## Three solve pipelines

- **`future`-Gauss**: classical Gauss-quadrature stiffness assembly
  (`PatchIntegrator.integrate_stiffness()`) + a direct sparse solve
  (`scipy.sparse.linalg.spsolve`). Not matrix-free, but the accuracy reference both
  matrix-free pipelines are checked against.
- **`future`-MF**: `WQMatrixFreeStiffness(patch, law).apply` (matrix-free WQ) +
  `FastDiagonalizationPreconditioner(..., law=law)` (auto-selects its `"mean"`
  method whenever a constitutive law is given) + GMRES via `future`'s own
  `matrix_free_solver.solve`.
- **`pymfiga`-MF**: `MechanicalModel.compute_mf_stiffness` (pymfiga's own matrix-free
  WQ operator) + its own fast-diagonalization preconditioner, driven through the
  *same* `matrix_free_solver.solve` GMRES driver as `future`-MF (pymfiga's own
  `solve_linearized_system` is a separate code path with its own `LinearSolver`
  class -- routing both matrix-free operators through one shared solver keeps the
  GMRES side of the comparison identical).

  `pymfiga`'s `LinearElasticity` only implements the plane-**strain** Lame formula
  (no plane-stress option). To still get the true plane-stress `(E, nu)` physics, it
  is fed the standard "apparent modulus" substitution
  `nu* = nu/(1+nu)`, `E* = E*(1+2*nu)/(1+nu)**2`, which makes its plane-strain
  formula reproduce the plane-stress constitutive matrix exactly (verified
  elsewhere in this repo to ~1e-15 relative Frobenius difference against `future`'s
  `PlaneStress`).

In [ ]:
def run_future_gauss_direct(pk, pdm, F, fixed, E, nu):
    law = PlaneStress(Material(E=E, nu=nu))
    ndof = 2 * pk.n_cp
    free = np.setdiff1d(np.arange(ndof), fixed)

    # basis_u/basis_v must stay alive in the same scope as the PatchIntegrator
    # using them: PatchIntegrator stores non-owning references (no pybind11
    # keep-alive), so once the Python objects that built them go out of scope
    # the references dangle.
    basis_u = IGABasis1D.build(pk.tensor.components[0], pk.tensor.components[0].degree + 1)
    basis_v = IGABasis1D.build(pk.tensor.components[1], pk.tensor.components[1].degree + 1)
    integrator = PatchIntegrator(pk, basis_u, basis_v, law)

    t0 = time.perf_counter()
    K = integrator.integrate_stiffness()
    t_assembly = time.perf_counter() - t0

    K_ff = K.tocsr()[free][:, free].tocsc()
    t0 = time.perf_counter()
    u = np.zeros(ndof)
    u[free] = spla.spsolve(K_ff, F[free])
    t_solve = time.perf_counter() - t0
    return dict(u=u, t_assembly=t_assembly, t_solve=t_solve, n_iter=1)


def run_future_matrix_free(pk, pdm, F, fixed, E, nu, rtol, maxiter, restart):
    law = PlaneStress(Material(E=E, nu=nu))
    ndof = 2 * pk.n_cp

    t0 = time.perf_counter()
    wq = WQMatrixFreeStiffness(pk, law)
    precond = FastDiagonalizationPreconditioner(pk, pdm, n_dofs_per_cp=2, fixed_dofs=fixed, law=law)
    t_assembly = time.perf_counter() - t0

    it = [0]

    def cb(_xk):
        it[0] += 1

    t0 = time.perf_counter()
    u, info = mf_solve(wq.apply, ndof, F, fixed, method="gmres",
                        precondition_fn=precond.apply, rtol=rtol, maxiter=maxiter,
                        callback=cb, restart=restart)
    t_solve = time.perf_counter() - t0
    if info != 0:
        raise RuntimeError(f"future matrix-free GMRES did not converge (info={info})")
    return dict(u=u, t_assembly=t_assembly, t_solve=t_solve, n_iter=it[0])


def run_pymfiga_matrix_free(pp, Rin, Tx, E, nu, rtol, maxiter, restart):
    n_cp = int(pp.nbctrlpts[0]) * int(pp.nbctrlpts[1])
    ndof = 2 * n_cp

    # pymfiga's LinearElasticity implements the plane-STRAIN Lame formula
    # unconditionally (no plane-stress option) -- feed it the apparent
    # (E*, nu*) that makes its plane-strain formula reproduce the true
    # plane-stress (E, nu) constitutive matrix exactly.
    nu_star = nu / (1 + nu)
    E_star = E * (1 + 2 * nu) / (1 + nu) ** 2

    t0 = time.perf_counter()
    material = LinearElasticity({"elastic_modulus": E_star, "poisson_ratio": nu_star})
    boundary = BoundaryCondition(nbctrlpts=pp.nbctrlpts, dofs=(DOF.UX, DOF.UY))
    boundary.add_constraint(
        constraint_info=[
            {"direction": ParametricDirection.ETA, "face": BoundarySide.MIN, "dofs": (DOF.UY,)},
            {"direction": ParametricDirection.ETA, "face": BoundarySide.MAX, "dofs": (DOF.UX,)},
        ],
        constraint_type="dirichlet",
    )
    model = MechanicalModel(material, pp, boundary)

    F = model.assemble_surface_force({
        DOF.ALL: ({"direction": ParametricDirection.XI, "face": BoundarySide.MAX}, kirsch_surface_force(Rin, Tx))
    })  # radial-max (r=Rex): outer arc, matching future's direction=0, side=1

    _free, fixed = model.get_free_and_constraint_nodes()
    fixed = np.array(sorted(fixed))

    precond = model.preconditioner
    # Geometry+material-aware correction (matches future's FastDiagonalizationPreconditioner
    # default method="mean" when law= is given): pymfiga computes this mean
    # coefficient itself but never wires it in unless asked.
    precond.add_scalar_space_time_correctors(
        mass_corrector=np.array(model.scalar_mean_mass).tolist(),
        stiffness_corrector=np.array(model.scalar_mean_stiffness).tolist(),
    )
    precond.update_space_eigenvalues((0.0, 1.0))  # A = 0*M + 1*K: pure stiffness preconditioner
    t_assembly = time.perf_counter() - t0

    it = [0]

    def cb(_xk):
        it[0] += 1

    # future's own matrix-free solver drives pymfiga's operator directly:
    # MechanicalModel.solve_linearized_system() is a separate code path with its
    # own LinearSolver class -- routing both matrix-free operators through one
    # shared GMRES driver keeps that half of the comparison identical.
    t0 = time.perf_counter()
    u_blocked, info = mf_solve(model.compute_mf_stiffness, ndof, F, fixed, method="gmres",
                               precondition_fn=precond.apply_spatial_preconditioner,
                               rtol=rtol, maxiter=maxiter, callback=cb, restart=restart)
    t_solve = time.perf_counter() - t0
    if info != 0:
        raise RuntimeError(f"pymfiga matrix-free GMRES did not converge (info={info})")

    u = np.empty(ndof)  # blocked [all ux, then all uy] -> interleaved [ux0,uy0,...]
    u[0::2] = u_blocked[:n_cp]
    u[1::2] = u_blocked[n_cp:]
    return dict(u=u, t_assembly=t_assembly, t_solve=t_solve, n_iter=it[0])


def compute_errors(pk, u, Rin, Tx, E, nu):
    basis_u = IGABasis1D.build(pk.tensor.components[0], pk.tensor.components[0].degree + 1)
    basis_v = IGABasis1D.build(pk.tensor.components[1], pk.tensor.components[1].degree + 1)
    law = PlaneStress(Material(E=E, nu=nu))
    integrator = PatchIntegrator(pk, basis_u, basis_v, law)
    u_exact_fn = lambda pt: kirsch_displacement(pt, Rin, Tx, E, nu)
    grad_exact_fn = lambda pt: kirsch_displacement_gradient(pt, Rin, Tx, E, nu)
    err_l2_sq = integrator.integrate_scalar_operator(L2ErrorOperator(u_exact_fn), u)
    err_h1_sq = integrator.integrate_scalar_operator(H1SemiNormErrorOperator(grad_exact_fn), u)
    return float(np.sqrt(err_l2_sq)), float(np.sqrt(err_l2_sq + err_h1_sq))

## Sanity check: do the three pipelines agree?

Before trusting a full sweep, run one concrete case (`degree=3, nbel=8`) through all
three pipelines and compare the L2/H1 error against the exact Kirsch solution --
these should agree closely, confirming `future`-MF and `pymfiga`-MF really solve the
*same* discrete problem as the Gauss reference, not just a similar one.

In [ ]:
def run_case(degree, nbel):
    """Builds the shared geometry once, runs all three pipelines, returns one
    result dict per pipeline with ndof/errors/timings/iteration count."""
    pp = build_pymfiga_patch(degree, nbel, Rin, Rex)
    pk, pdm = build_future_from_pymfiga(pp)
    ndof = 2 * pk.n_cp
    fixed = future_fixed_dofs(pk, pdm)

    basis_u = IGABasis1D.build(pk.tensor.components[0], pk.tensor.components[0].degree + 1)
    basis_v = IGABasis1D.build(pk.tensor.components[1], pk.tensor.components[1].degree + 1)
    law = PlaneStress(Material(E=E, nu=nu))
    F = PatchIntegrator(pk, basis_u, basis_v, law).integrate_boundary_load(0, 1, KirschTraction(Rin, Tx))

    pipelines = {
        "future-gauss": lambda: run_future_gauss_direct(pk, pdm, F, fixed, E, nu),
        "future-mf": lambda: run_future_matrix_free(pk, pdm, F, fixed, E, nu, rtol=1e-10, maxiter=3000, restart=200),
        "pymfiga-mf": lambda: run_pymfiga_matrix_free(pp, Rin, Tx, E, nu, rtol=1e-10, maxiter=3000, restart=200),
    }
    rows = []
    for name, run_fn in pipelines.items():
        r = run_fn()
        l2_error, h1_error = compute_errors(pk, r["u"], Rin, Tx, E, nu)
        rows.append(dict(degree=degree, nbel=nbel, pipeline=name, ndof=ndof,
                          l2_error=l2_error, h1_error=h1_error,
                          t_assembly=r["t_assembly"], t_solve=r["t_solve"],
                          n_iter=r["n_iter"]))
    return rows


sanity_rows = run_case(degree=3, nbel=8)
print(f"{'pipeline':<14} {'ndof':>6} {'l2_error':>12} {'h1_error':>12} "
      f"{'t_assembly':>11} {'t_solve':>9} {'n_iter':>7}")
for row in sanity_rows:
    print(f"{row['pipeline']:<14} {row['ndof']:>6d} {row['l2_error']:>12.4e} "
          f"{row['h1_error']:>12.4e} {row['t_assembly']:>10.4f}s {row['t_solve']:>8.4f}s "
          f"{row['n_iter']:>7d}")

h1_errors = {row["pipeline"]: row["h1_error"] for row in sanity_rows}
ref = h1_errors["future-gauss"]
for name, val in h1_errors.items():
    rel_diff = abs(val - ref) / ref
    assert rel_diff < 1e-3, f"{name}: H1 error differs from future-gauss by {rel_diff:.2%}"
    print(f"{name}: H1 error within {rel_diff:.2%} of the future-gauss reference")

## The sweep: 3 degrees x 7 element counts, 3 pipelines

For every `(degree, nbel)`, all three pipelines solve the same discretization and
are checked against the true closed-form Kirsch solution -- so the resulting plot
shows IGA's optimal convergence rate for every pipeline, not just agreement between
them.

In [ ]:
degrees = [2, 3, 4]
nbels = [1, 2, 4, 8, 16, 32, 64]
pipelines_order = ["future-gauss", "future-mf", "pymfiga-mf"]

results = []
print(f"{'p':>2} {'nbel':>4}  " + "  ".join(f"{name:>12}" for name in pipelines_order))
for degree in degrees:
    for nbel in nbels:
        rows = run_case(degree, nbel)
        results.extend(rows)
        h1_by_pipeline = {r["pipeline"]: r["h1_error"] for r in rows}
        print(f"{degree:>2} {nbel:>4d}  "
              + "  ".join(f"{h1_by_pipeline[name]:>12.3e}" for name in pipelines_order))

print(f"\n{len(results)} rows collected.")

## A closer look: GMRES residual history at `degree=6, nbel=64`

The sweep above only reports the *final* GMRES iteration count, at a single,
fixed preconditioner setting (`future`'s `FastDiagonalizationPreconditioner`
with its default `"mean"` method). To see *how* that preconditioner actually
helps, this section reproduces the kind of residual-history plot J. Cornejo's
thesis uses (Fig. 2.10, p.73) -- relative residual norm versus GMRES iteration
-- at `degree=6, nbel=64` (the same discretization the thesis uses there), for
three settings of `future`'s own preconditioner:

- **no preconditioner** -- plain GMRES on `WQMatrixFreeStiffness.apply`;
- **classic FD** -- `FastDiagonalizationPreconditioner(..., method="classic")`,
  the geometry/material-agnostic fast-diagonalization preconditioner;
- **mean FD** -- `FastDiagonalizationPreconditioner(..., law=law,
  method="mean")`, the material-aware correction (thesis Eq. 2.15) used
  everywhere else in this notebook.

The relative residual at each iteration is read directly from GMRES itself via
`callback_type="pr_norm"` (scipy's preconditioned-relative-residual-norm
callback -- for the no-preconditioner run this is just the plain relative
residual, since there is no preconditioning to apply).

In [ ]:
import math

TARGET_TOL = 1e-12


def residual_history(pk, pdm, F, fixed, law, method, max_steps, restart=50):
    """Relative residual norm at every GMRES iteration, for one preconditioner
    setting. `method` is None (no preconditioner), "classic", or "mean".

    A single long, non-restarted cycle (restart=max_steps, maxiter=1) looks
    like the natural way to cap the total iteration count at `max_steps`, but
    it misbehaves here: once the Arnoldi process driving that one cycle hits a
    Krylov breakdown (the preconditioned residual gets so small the next
    search direction is numerically zero), the cycle just stops -- and scipy
    only re-checks the convergence criterion at a restart boundary, so even a
    residual already below `rtol` can come back as "not converged". Splitting
    the same total budget into several smaller `restart`-sized cycles
    (`maxiter = ceil(max_steps / restart)`) fixes both problems at once: each
    cycle boundary is a fresh convergence check, AND restarting rebuilds the
    Krylov basis from a freshly computed residual, which lets GMRES carry on
    past a breakdown that would have stalled a single long cycle.

    scipy's own stopping decision can still lag behind what the callback
    reports -- e.g. "mean" crosses TARGET_TOL at iteration 37 but scipy
    doesn't actually stop until iteration 51, chasing the residual another 3
    orders of magnitude down to ~8e-16 for no practical benefit. Since there's
    nothing useful left to show once the target has been reached, the
    returned history is trimmed right at the first iteration that reaches
    TARGET_TOL, rather than plotting/reporting that extra tail."""
    ndof = 2 * pk.n_cp
    wq = WQMatrixFreeStiffness(pk, law)
    precond_fn = None
    if method is not None:
        precond = FastDiagonalizationPreconditioner(pk, pdm, n_dofs_per_cp=2,
                                                     fixed_dofs=fixed, law=law, method=method)
        precond_fn = precond.apply

    residuals = []
    def cb(res):
        residuals.append(res)

    maxiter = math.ceil(max_steps / restart)
    try:
        mf_solve(wq.apply, ndof, F, fixed, method="gmres", precondition_fn=precond_fn,
                 rtol=1e-12, maxiter=maxiter, restart=restart,
                 callback=cb, callback_type="pr_norm")
    except RuntimeError:
        pass  # did not reach rtol within max_steps -- the partial history is still useful

    for idx, r in enumerate(residuals):
        if r <= TARGET_TOL:
            return residuals[:idx + 1]
    return residuals


degree, nbel = 6, 64
pp = build_pymfiga_patch(degree, nbel, Rin, Rex)
pk, pdm = build_future_from_pymfiga(pp)
fixed = future_fixed_dofs(pk, pdm)
basis_u = IGABasis1D.build(pk.tensor.components[0], pk.tensor.components[0].degree + 1)
basis_v = IGABasis1D.build(pk.tensor.components[1], pk.tensor.components[1].degree + 1)
law = PlaneStress(Material(E=E, nu=nu))
F_case = PatchIntegrator(pk, basis_u, basis_v, law).integrate_boundary_load(0, 1, KirschTraction(Rin, Tx))

MAX_STEPS = 150
histories = {
    "no preconditioner": residual_history(pk, pdm, F_case, fixed, law, None, MAX_STEPS),
    "classic FD": residual_history(pk, pdm, F_case, fixed, law, "classic", MAX_STEPS),
    "mean FD": residual_history(pk, pdm, F_case, fixed, law, "mean", MAX_STEPS),
}

print(f"degree={degree}, nbel={nbel}, ndof={2 * pk.n_cp}")
for name, hist in histories.items():
    if hist:
        print(f"  {name:<20} {len(hist):>4d} iterations, final relative residual = {hist[-1]:.3e}")
    else:
        print(f"  {name:<20} no residual recorded")

fig, ax = plt.subplots(figsize=(7, 5))
for name, hist in histories.items():
    ax.semilogy(range(1, len(hist) + 1), hist, marker=".", markersize=3, label=name)
ax.set_xlabel("GMRES iteration")
ax.set_ylabel("relative residual")
ax.set_title(f"Residual history, degree={degree}, nbel={nbel} (ndof={2 * pk.n_cp})")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

**Why 50-iteration restart cycles instead of one long run?** A first attempt
used a single, non-restarted cycle covering the whole 150-iteration budget
(`restart=150, maxiter=1`). It looked reasonable, but `classic` FD stalled at a
relative residual of ~1.8e-10 after only 71 steps, with GMRES reporting
`info=1` ("not converged") even though plenty of budget was left: once the
residual got that low, the Arnoldi process driving that one long cycle hit a
**Krylov breakdown** (floating-point noise made the next search direction
numerically zero) and had nowhere left to go within that single cycle. Splitting
the same 150-iteration budget into 50-iteration restart cycles fixes this:
restarting rebuilds the Krylov basis from a freshly computed residual, letting
GMRES carry on past the point where the old single cycle got stuck. With
`restart=50`, `classic` FD now genuinely converges (`info=0`) to `rtol=1e-12`
in 98 iterations. `no preconditioner` is unaffected by the restart strategy
either way: it is nowhere near converging within a 150-iteration budget
regardless of how that budget is split (it needs on the order of 10,000
iterations to actually reach `rtol=1e-12`, confirmed separately).

One more wrinkle, specific to `mean` FD: scipy's own stopping decision can lag
behind what the callback reports. `mean`'s residual crosses `TARGET_TOL=1e-12`
at iteration 37, but GMRES doesn't actually declare convergence until iteration
51, having chased the residual another three orders of magnitude down to ~8e-16
in the meantime -- precision nobody asked for. There's nothing useful left to
show once the target is reached, so `residual_history` trims the returned curve
at the first iteration that reaches `TARGET_TOL`, rather than plotting (or
reporting an iteration count for) that extra, pointless tail -- `mean` FD is
reported as **37 iterations**, not 51.

## GMRES vs BiCGStab

`matrix_free_solver.solve` also supports `method="bicgstab"` as a drop-in
alternative to `"gmres"` -- unlike `"cg"` (which requires exact symmetry and can
fail outright on `WQMatrixFreeStiffness.apply()`'s approximate symmetry on curved
geometry), BiCGStab does not assume symmetry, so it should not share that failure
mode. It also needs no `restart` parameter: its per-iteration memory is constant,
unlike GMRES's growing Krylov basis.

The comparison isn't quite apples-to-apples on raw iteration count, though:
BiCGStab does **two** matrix-free operator applies (and two preconditioner
applies) per iteration, against GMRES's one -- so wall-clock time, not iteration
count, is the fair comparison. BiCGStab's scipy callback also only hands back
the current iterate (not a residual norm the way GMRES's `callback_type="pr_norm"`
does), so its relative residual below is recomputed by hand at each iteration via
one extra `apply()` call.

In [ ]:
def residual_history_bicgstab(pk, pdm, F, fixed, law, method, max_steps):
    """Like residual_history, but for BiCGStab: its callback only hands back
    the current iterate xk, not a residual norm, so the relative residual is
    recomputed by hand at each iteration via one extra matrix-free apply."""
    ndof = 2 * pk.n_cp
    wq = WQMatrixFreeStiffness(pk, law)
    precond_fn = None
    if method is not None:
        precond = FastDiagonalizationPreconditioner(pk, pdm, n_dofs_per_cp=2,
                                                     fixed_dofs=fixed, law=law, method=method)
        precond_fn = precond.apply

    op, free_dofs = free_dof_operator(wq.apply, ndof, fixed)
    rhs = F[free_dofs]
    rhs_norm = np.linalg.norm(rhs)

    residuals = []
    def cb(xk):
        residuals.append(np.linalg.norm(rhs - op.matvec(xk)) / rhs_norm)

    mf_solve(wq.apply, ndof, F, fixed, method="bicgstab", precondition_fn=precond_fn,
             rtol=1e-12, maxiter=max_steps, callback=cb)

    for idx, r in enumerate(residuals):
        if r <= TARGET_TOL:
            return residuals[:idx + 1]
    return residuals


solver_comparison = {}
for precond_method in (None, "classic", "mean"):
    label = precond_method if precond_method else "no preconditioner"

    t0 = time.perf_counter()
    hist_gmres = residual_history(pk, pdm, F_case, fixed, law, precond_method, MAX_STEPS)
    t_gmres = time.perf_counter() - t0

    t0 = time.perf_counter()
    hist_bicgstab = residual_history_bicgstab(pk, pdm, F_case, fixed, law, precond_method, MAX_STEPS)
    t_bicgstab = time.perf_counter() - t0

    solver_comparison[label] = dict(gmres=(hist_gmres, t_gmres), bicgstab=(hist_bicgstab, t_bicgstab))

print(f"{'setting':<20} {'solver':<10} {'n_iter':>7} {'final_res':>12} {'time':>8}")
for label, d in solver_comparison.items():
    for solver_name, (hist, t) in d.items():
        final_res = hist[-1] if hist else float("nan")
        print(f"{label:<20} {solver_name:<10} {len(hist):>7d} {final_res:>12.3e} {t:>7.3f}s")

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (label, d) in zip(axes, solver_comparison.items()):
    for solver_name, (hist, t) in d.items():
        ax.semilogy(range(1, len(hist) + 1), hist, marker=".", markersize=3,
                    label=f"{solver_name} ({t:.2f}s)")
    ax.set_xlabel("iteration")
    ax.set_title(label)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("relative residual")
fig.suptitle(f"GMRES vs BiCGStab, degree={degree}, nbel={nbel} (ndof={2 * pk.n_cp})")
fig.tight_layout()
plt.show()

## Reading the numbers

- **Accuracy**: at every `(degree, nbel)` in the sweep, `future-mf` and
  `pymfiga-mf` give the *same* H1 error as each other (matching to the last
  printed digit) and track the `future`-Gauss reference closely -- e.g. at
  `degree=3, nbel=8`: `future-gauss=1.048e-06`, `future-mf=1.048e-06`,
  `pymfiga-mf=1.048e-06`. Both matrix-free implementations really do reproduce
  the classical Gauss-quadrature discrete solution, not an approximation of it.
- **Convergence**: the H1 error drops at the expected optimal IGA rate for each
  degree (e.g. degree 3: `7.1e-05 (nbel=2) -> 1.0e-05 (4) -> 1.0e-06 (8) ->
  1.3e-07 (16) -> 1.7e-08 (32) -> 2.2e-09 (64)`), identically for all three
  pipelines.
- **GMRES iteration count** is essentially flat across the whole sweep (~40-43
  iterations for both matrix-free pipelines, regardless of degree or `nbel`) --
  the fast-diagonalization preconditioner is doing its job on both sides.
- **Timing**: despite near-identical iteration counts, `future-mf`'s solve
  phase is consistently 2-2.5x faster than `pymfiga-mf`'s at the same
  `(degree, nbel)` -- e.g. at `degree=4, nbel=64`: `future-mf` solves in 0.67s
  versus `pymfiga-mf`'s 1.58s. The per-iteration cost of `future`'s matrix-free
  operator and preconditioner apply is noticeably cheaper than `pymfiga`'s
  implementation of the same algorithm.
- **Matrix-free vs Gauss crossover**: at small problems (`degree=3, nbel=8`,
  242 dofs) the direct Gauss solve is still faster overall (0.017s total versus
  `future-mf`'s 0.027s) -- matrix-free's setup cost isn't amortized yet. At
  `degree=4, nbel=64` (9248 dofs) the picture flips sharply: `future-gauss`
  takes 3.6s total (assembly+solve) against `future-mf`'s 0.85s, a ~4.2x
  speedup -- Gauss assembly and the direct factorization both grow much faster
  with degree than the matrix-free operator does.

In [ ]:
colors = {2: "tab:blue", 3: "tab:orange", 4: "tab:green"}
linestyles = {"future-gauss": "-", "future-mf": "--", "pymfiga-mf": ":"}
markers = {"future-gauss": "o", "future-mf": "s", "pymfiga-mf": "^"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for degree in degrees:
    for pipeline in pipelines_order:
        rows = [r for r in results if r["degree"] == degree and r["pipeline"] == pipeline]
        rows.sort(key=lambda r: r["nbel"])
        nbel_vals = [r["nbel"] for r in rows]
        h1_vals = [r["h1_error"] for r in rows]
        time_vals = [r["t_assembly"] + r["t_solve"] for r in rows]

        style = dict(color=colors[degree], linestyle=linestyles[pipeline],
                     marker=markers[pipeline], label=f"{pipeline}, p={degree}")
        axes[0].loglog(nbel_vals, h1_vals, **style)
        axes[1].loglog(nbel_vals, time_vals, **style)
        if pipeline != "future-gauss":  # n_iter==1 (direct solve) isn't a GMRES count
            iter_vals = [r["n_iter"] for r in rows]
            axes[2].semilogx(nbel_vals, iter_vals, **style)

axes[0].set_xlabel("elements / side"); axes[0].set_ylabel("H1 error"); axes[0].set_title("Accuracy")
axes[1].set_xlabel("elements / side"); axes[1].set_ylabel("assembly + solve time [s]"); axes[1].set_title("Wall-clock time")
axes[2].set_xlabel("elements / side"); axes[2].set_ylabel("GMRES iterations"); axes[2].set_title("Iteration count (GMRES pipelines only)")
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
axes[0].legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## Summary

- `future` and `pymfiga`'s matrix-free WQ stiffness operators, run through the
  same GMRES + fast-diagonalization-preconditioner driver, reproduce the
  classical Gauss-quadrature solution to the exact Kirsch benchmark at every
  degree and mesh size tested (`degree` 2-4, up to 64 elements/side) --
  correctness is not in question for either implementation.
- `future`'s implementation is consistently faster per GMRES iteration than
  `pymfiga`'s (2-2.5x on the solve phase at the sizes tested here), with
  iteration counts themselves essentially identical between the two.
- Matrix-free only pays off once the problem is large enough and the degree
  high enough that Gauss assembly + direct factorization cost dominates; at
  the smallest sizes tested, the direct solve is still faster.